# Predikcia dojazdu elektrických áut: preprocessing

Pokračujeme z notebooku `analyza.ipynb`, ktorý vyčistil dáta a uložil ich do `data/ev_clean.csv`.
Tu dáta pripravíme na trénovanie modelu.

## 1. Načítanie vyčistených dát

In [1]:
import pandas as pd

data = pd.read_csv('data/ev_clean.csv')
print(data.shape)
data.head()

(1585, 9)


,year,make,vehicle_class,drive,motor_kw,n_motors,charge_time_240v_hr,kwh_per_100km,range_km
0,2011,Nissan,Midsize Cars,Front-Wheel Drive,80.0,1,7.0,21.13,117.0
1,2011,smart,Two Seaters,Rear-Wheel Drive,30.0,1,8.0,24.23,101.0
2,2011,smart,Two Seaters,Rear-Wheel Drive,30.0,1,8.0,24.23,101.0
3,2012,Mitsubishi,Subcompact Cars,Rear-Wheel Drive,49.0,1,7.0,18.64,100.0
4,2012,Azure Dynamics,Special Purpose Vehicle 2WD,Front-Wheel Drive,52.0,1,8.0,33.55,90.0


In [2]:
# kontrola, že čistenie prebehlo: žiadne chýbajúce hodnoty
print('Chýbajúce hodnoty:', data.isna().sum().sum())

Chýbajúce hodnoty: 0


## 2. Rozdelenie na X a y

- **`y`** je cieľová premenná, ktorú chceme predpovedať: dojazd `range_km`.
- **`X`** sú príznaky, z ktorých ju model predpovedá: všetky ostatné stĺpce.

In [3]:
y = data['range_km']
X = data.drop(columns='range_km')

print('X:', X.shape)
print('y:', y.shape)

X: (1585, 8)
y: (1585,)


In [4]:
# príznaky rozdelíme podľa typu, každý typ budeme spracovávať inak
num_col = X.select_dtypes(include='number').columns.tolist()
cat_col = X.select_dtypes(exclude='number').columns.tolist()

print('Číselné príznaky:', num_col)
print('Kategorické príznaky:', cat_col)

Číselné príznaky: ['year', 'motor_kw', 'n_motors', 'charge_time_240v_hr', 'kwh_per_100km']
Kategorické príznaky: ['make', 'vehicle_class', 'drive']


In [5]:
X.head()

,year,make,vehicle_class,drive,motor_kw,n_motors,charge_time_240v_hr,kwh_per_100km
0,2011,Nissan,Midsize Cars,Front-Wheel Drive,80.0,1,7.0,21.13
1,2011,smart,Two Seaters,Rear-Wheel Drive,30.0,1,8.0,24.23
2,2011,smart,Two Seaters,Rear-Wheel Drive,30.0,1,8.0,24.23
3,2012,Mitsubishi,Subcompact Cars,Rear-Wheel Drive,49.0,1,7.0,18.64
4,2012,Azure Dynamics,Special Purpose Vehicle 2WD,Front-Wheel Drive,52.0,1,8.0,33.55


In [6]:
y.describe().round(1)

count    1585.0
mean      447.8
std       120.7
min        61.0
25%       389.0
50%       455.0
75%       507.0
max       837.0
Name: range_km, dtype: float64

V `X` je 8 príznakov: 5 číselných (`year`, `motor_kw`, `n_motors`, `charge_time_240v_hr`,
`kwh_per_100km`) a 3 kategorické (`make`, `vehicle_class`, `drive`). Kategorické príznaky
v ďalšom kroku zakódujeme cez one-hot encoding, číselné podľa typu modelu škálujeme.